# Diagnose V028 grid rejection using its original training-only pool
No controller equations are fitted, and no validation/test pressures or model scores are computed. This exploratory analysis preserves the frozen failed result.

In [1]:
from pathlib import Path
import json,hashlib
import numpy as np,pandas as pd
P=Path.cwd();O=P/'outputs/controller_quantization_feasibility_v028_review'
with np.load(P/'outputs/improvements_v005/sequence/split_breath_ids.npz') as z:fit=z['A_fit']
sample=json.loads((P/'outputs/controller_quantization_feasibility_v028/sample_before_scoring.json').read_text());excluded=set(sample['primary_ids']+sample['replicate_ids']);pool=set(map(int,fit))-excluded
meta=pd.read_csv(P/'data/train.csv',usecols=['breath_id'],dtype={'breath_id':'int64'}).iloc[::80].reset_index(drop=True);allowed=set(np.flatnonzero(meta.breath_id.isin(pool).to_numpy()).tolist())
df=pd.read_csv(P/'data/train.csv',usecols=['pressure','u_out'],dtype={'pressure':'float64','u_out':'int8'},skiprows=lambda i:i>0 and (i-1)//80 not in allowed,float_precision='round_trip')
v=np.unique(df.pressure.to_numpy()[df.u_out.to_numpy()==0]);magnitude=max(1.,float(np.max(abs(v))));tol=4096*np.finfo(float).eps*magnitude;gaps=np.diff(v);gaps=gaps[gaps>64*np.finfo(float).eps*magnitude];nominal=float(np.median(gaps));codes=np.rint((v-v[0])/nominal);q,origin=np.linalg.lstsq(np.column_stack([codes,np.ones(len(codes))]),v,rcond=None)[0];res=abs(v-(origin+q*codes))
source='\n'.join(''.join(c['source']) for c in json.loads((P/'research/quantized_controller_components_v028.ipynb').read_text())['cells'] if c['cell_type']=='code');ns={};exec(compile(source,'q28-components','exec'),ns)
base=np.arange(100)*.07;aliases=np.r_[base,np.nextafter(base,np.inf),np.nextafter(np.nextafter(base,np.inf),np.inf)];toy_rejected=False
try:ns['q28_lattice'](aliases)
except ValueError:toy_rejected=True
r=dict(raw_unique_levels=len(v),float32_unique_levels=len(np.unique(v.astype(np.float32))),integer_level_count=len(np.unique(codes)),nominal_spacing=nominal,fitted_spacing=float(q),maximum_lattice_residual=float(res.max()),median_lattice_residual=float(np.median(res)),declared_roundoff_bound=tol,raw_adjacent_code_fraction=float(np.mean(np.diff(codes)==1)),distinct_adjacent_code_fraction=float(np.mean(np.diff(np.unique(codes))==1)),raw_exact_float32_fraction=float(np.mean(v.astype(np.float32).astype(np.float64)==v)),maximum_float32_roundtrip_error=float(np.max(abs(v-v.astype(np.float32).astype(np.float64)))),toy_roundoff_aliases_rejected=toy_rejected,scope='Exploratory training-pool representation diagnosis only, no controller fit, nativepressureMAE or promotion; V028 remains stopped')
with (O/'grid_diagnosis.json').open('x') as f:f.write(json.dumps(r,indent=2)+'\n')
print(json.dumps(r,indent=2))


{
  "raw_unique_levels": 950,
  "float32_unique_levels": 950,
  "integer_level_count": 950,
  "nominal_spacing": 0.07030214599999951,
  "fitted_spacing": 0.07030214545140898,
  "maximum_lattice_residual": 5.025711402595334e-09,
  "median_lattice_residual": 1.943639915680251e-09,
  "declared_roundoff_bound": 5.895434855119674e-11,
  "raw_adjacent_code_fraction": 1.0,
  "distinct_adjacent_code_fraction": 1.0,
  "raw_exact_float32_fraction": 0.0,
  "maximum_float32_roundtrip_error": 3.590058597069401e-06,
  "toy_roundoff_aliases_rejected": true,
  "scope": "Exploratory training-pool representation diagnosis only, no controller fit, nativepressureMAE or promotion; V028 remains stopped"
}
